# 运行本地服务器

本指南介绍如何在本地运行 LangGraph 应用。

## 前置条件

开始之前, 请确保你具备以下条件:

* 一个 [LangSmith](https://smith.langchain.com/settings) API key。

## 1. 安装 LangGraph CLI

```bash
# Python >= 3.11 is required.
pip install -U "langgraph-cli[inmem]"
```

```bash
# Python >= 3.11 is required.
uv add "langgraph-cli[inmem]"
```

## 2. 创建 LangGraph 应用

基于 [`new-langgraph-project-python` 模板](https://github.com/langchain-ai/new-langgraph-project) 创建一个新应用。该模板演示了一个单 node 应用, 你可以在其基础上用自己的逻辑进行扩展。

```shell
langgraph new path/to/your/app --template new-langgraph-project-python
```

> **提示**
>
> **更多模板**: 如果你使用 `langgraph new` 时不指定模板, 会看到一个交互式菜单, 你可以从可用模板列表中进行选择。

## 3. 安装依赖

在新 LangGraph 应用的根目录下, 以 `edit` 模式安装依赖, 这样服务器才会使用你的本地改动:

```bash
cd path/to/your/app
pip install -e .
```

```bash
cd path/to/your/app
uv sync
```

## 4. 创建 `.env` 文件

你会在新 LangGraph 应用的根目录下看到一个 `.env.example`。请在该根目录创建 `.env` 文件, 把 `.env.example` 的内容复制进去, 并填入所需的 API key:

```bash
LANGSMITH_API_KEY=lsv2...
```

## 5. 启动 Agent server

在本地启动 LangGraph API 服务器:

```shell
langgraph dev
```

示例输出:

```
INFO:langgraph_api.cli:

        Welcome to

╦  ┌─┐┌┐┌┌─┐╔═╗┬─┐┌─┐┌─┐┬ ┬
║  ├─┤││││ ┬║ ╦├┬┘├─┤├─┘├─┤
╩═╝┴ ┴┘└┘└─┘╚═╝┴└─┴ ┴┴  ┴ ┴

- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs

This in-memory server is designed for development and testing.
For production use, please use LangSmith Deployment.
```

`langgraph dev` 命令会以内存模式启动 Agent Server。该模式适合开发与测试。生产环境请部署可访问持久化存储后端的 Agent Server。更多信息请参阅 [Platform setup overview](https://docs.langchain.com/langsmith/platform-setup)。

## 6. 在 Studio 中测试你的应用

[Studio](https://docs.langchain.com/langsmith/studio) 是一个专用 UI, 你可以把它连接到 LangGraph API 服务器, 在本地可视化、交互并调试你的应用。访问 `langgraph dev` 命令输出中提供的 URL, 即可在 Studio 中测试你的 graph:

```
>    - LangGraph Studio Web UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
```

如果 Agent Server 运行在自定义 host/port 上, 请更新 URL 中的 `baseUrl` 查询参数。例如, 服务器运行在 `http://myhost:3000` 时:

```
https://smith.langchain.com/studio/?baseUrl=http://myhost:3000
```

> **提示**
>
> **Safari 兼容性**: 请在命令中加上 `--tunnel` 标志以创建安全隧道, 因为 Safari 在连接 localhost 服务器时存在限制:
>
> ```shell
> langgraph dev --tunnel
> ```

## 7. 测试 API

### Python SDK (async)

先安装 LangGraph Python SDK:

```shell
pip install langgraph-sdk
```

然后向 assistant 发送一条消息(无 thread 运行):

```python
from langgraph_sdk import get_client
import asyncio

client = get_client(url="http://localhost:2024")

async def main():
    async for chunk in client.runs.stream(
        None,  # Threadless run
        "agent", # Name of assistant. Defined in langgraph.json.
        input={
        "messages": [{
            "role": "human",
            "content": "What is LangGraph?",
            }],
        },
    ):
        print(f"Receiving new event of type: {chunk.event}...")
        print(chunk.data)
        print("\n\n")

asyncio.run(main())
```

> **说明**
>
> 上面的 async 片段通过 `asyncio.run(main())` 驱动, 而 `asyncio.run()` 不能在已经运行事件循环的环境中调用(例如 Jupyter notebook 的 kernel), 因此这里保留为代码块而不参与执行。同步版本的等价用法见下一节。

### Python SDK (sync)

先安装 LangGraph Python SDK:

```shell
pip install langgraph-sdk
```

然后向 assistant 发送一条消息(无 thread 运行):

In [1]:
from langgraph_sdk import get_sync_client

client = get_sync_client(url="http://localhost:2024")

for chunk in client.runs.stream(
    None,  # Threadless run
    "agent", # Name of assistant. Defined in langgraph.json.
    input={
        "messages": [{
            "role": "human",
            "content": "What is LangGraph?",
        }],
    },
    stream_mode="messages-tuple",
):
    print(f"Receiving new event of type: {chunk.event}...")
    print(chunk.data)
    print("\n\n")

Receiving new event of type: metadata...
{'run_id': '01a0f852-2437-7b23-ac74-f26e05568109', 'attempt': 1}



Receiving new event of type: messages...
[{'content': '', 'additional_kwargs': {}, 'response_metadata': {'model_provider': 'deepseek'}, 'type': 'AIMessageChunk', 'name': None, 'id': 'lc_run--01a0f852-281c-7401-af4f-eeea5aefc542', 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata': None, 'tool_call_chunks': [], 'chunk_position': None}, {'created_by': 'system', 'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca', 'run_attempt': 1, 'langgraph_version': '1.2.12', 'langgraph_api_version': '0.17.0.dev3', 'langgraph_plan': 'enterprise', 'langgraph_host': 'self-hosted', 'langgraph_api_url': 'http://127.0.0.1:2024', 'run_id': '01a0f852-2437-7b23-ac74-f26e05568109', 'thread_id': '01a0f852-2437-7b23-ac74-f27e6db23df3', 'graph_id': 'agent', 'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'call_model', 'langgraph_triggers': ['branch:to:call_model

Receiving new event of type: messages...
[{'content': '**', 'additional_kwargs': {}, 'response_metadata': {'model_provider': 'deepseek'}, 'type': 'AIMessageChunk', 'name': None, 'id': 'lc_run--01a0f852-281c-7401-af4f-eeea5aefc542', 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata': None, 'tool_call_chunks': [], 'chunk_position': None}, {'created_by': 'system', 'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca', 'run_attempt': 1, 'langgraph_version': '1.2.12', 'langgraph_api_version': '0.17.0.dev3', 'langgraph_plan': 'enterprise', 'langgraph_host': 'self-hosted', 'langgraph_api_url': 'http://127.0.0.1:2024', 'run_id': '01a0f852-2437-7b23-ac74-f26e05568109', 'thread_id': '01a0f852-2437-7b23-ac74-f27e6db23df3', 'graph_id': 'agent', 'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'call_model', 'langgraph_triggers': ['branch:to:call_model'], 'langgraph_path': ['__pregel_pull', 'call_model'], 'langgraph_checkpoint_ns': 'call_model:3c9d4839-5b9d

Receiving new event of type: messages...
[{'content': 'ful', 'additional_kwargs': {}, 'response_metadata': {'model_provider': 'deepseek'}, 'type': 'AIMessageChunk', 'name': None, 'id': 'lc_run--01a0f852-281c-7401-af4f-eeea5aefc542', 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata': None, 'tool_call_chunks': [], 'chunk_position': None}, {'created_by': 'system', 'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca', 'run_attempt': 1, 'langgraph_version': '1.2.12', 'langgraph_api_version': '0.17.0.dev3', 'langgraph_plan': 'enterprise', 'langgraph_host': 'self-hosted', 'langgraph_api_url': 'http://127.0.0.1:2024', 'run_id': '01a0f852-2437-7b23-ac74-f26e05568109', 'thread_id': '01a0f852-2437-7b23-ac74-f27e6db23df3', 'graph_id': 'agent', 'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'call_model', 'langgraph_triggers': ['branch:to:call_model'], 'langgraph_path': ['__pregel_pull', 'call_model'], 'langgraph_checkpoint_ns': 'call_model:3c9d4839-5b9

Receiving new event of type: messages...
[{'content': 'like', 'additional_kwargs': {}, 'response_metadata': {'model_provider': 'deepseek'}, 'type': 'AIMessageChunk', 'name': None, 'id': 'lc_run--01a0f852-281c-7401-af4f-eeea5aefc542', 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata': None, 'tool_call_chunks': [], 'chunk_position': None}, {'created_by': 'system', 'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca', 'run_attempt': 1, 'langgraph_version': '1.2.12', 'langgraph_api_version': '0.17.0.dev3', 'langgraph_plan': 'enterprise', 'langgraph_host': 'self-hosted', 'langgraph_api_url': 'http://127.0.0.1:2024', 'run_id': '01a0f852-2437-7b23-ac74-f26e05568109', 'thread_id': '01a0f852-2437-7b23-ac74-f27e6db23df3', 'graph_id': 'agent', 'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'call_model', 'langgraph_triggers': ['branch:to:call_model'], 'langgraph_path': ['__pregel_pull', 'call_model'], 'langgraph_checkpoint_ns': 'call_model:3c9d4839-5b

Receiving new event of type: messages...
[{'content': 'can', 'additional_kwargs': {}, 'response_metadata': {'model_provider': 'deepseek'}, 'type': 'AIMessageChunk', 'name': None, 'id': 'lc_run--01a0f852-281c-7401-af4f-eeea5aefc542', 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata': None, 'tool_call_chunks': [], 'chunk_position': None}, {'created_by': 'system', 'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca', 'run_attempt': 1, 'langgraph_version': '1.2.12', 'langgraph_api_version': '0.17.0.dev3', 'langgraph_plan': 'enterprise', 'langgraph_host': 'self-hosted', 'langgraph_api_url': 'http://127.0.0.1:2024', 'run_id': '01a0f852-2437-7b23-ac74-f26e05568109', 'thread_id': '01a0f852-2437-7b23-ac74-f27e6db23df3', 'graph_id': 'agent', 'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'call_model', 'langgraph_triggers': ['branch:to:call_model'], 'langgraph_path': ['__pregel_pull', 'call_model'], 'langgraph_checkpoint_ns': 'call_model:3c9d4839-5b9

...

Receiving new event of type: messages...
[{'content': ')\n\n', 'additional_kwargs': {}, 'response_metadata': {'model_provider': 'deepseek'}, 'type': 'AIMessageChunk', 'name': None, 'id': 'lc_run--01a0f852-281c-7401-af4f-eeea5aefc542', 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata': None, 'tool_call_chunks': [], 'chunk_position': None}, {'created_by': 'system', 'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca', 'run_attempt': 1, 'langgraph_version': '1.2.12', 'langgraph_api_version': '0.17.0.dev3', 'langgraph_plan': 'enterprise', 'langgraph_host': 'self-hosted', 'langgraph_api_url': 'http://127.0.0.1:2024', 'run_id': '01a0f852-2437-7b23-ac74-f26e05568109', 'thread_id': '01a0f852-2437-7b23-ac74-f27e6db23df3', 'graph_id': 'agent', 'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'call_model', 'langgraph_triggers': ['branch:to:call_model'], 'langgraph_path': ['__pregel_pull', 'call_model'], 'langgraph_checkpoint_ns': 'call_model:3c9d4839-5

Receiving new event of type: messages...
[{'content': '"]', 'additional_kwargs': {}, 'response_metadata': {'model_provider': 'deepseek'}, 'type': 'AIMessageChunk', 'name': None, 'id': 'lc_run--01a0f852-281c-7401-af4f-eeea5aefc542', 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata': None, 'tool_call_chunks': [], 'chunk_position': None}, {'created_by': 'system', 'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca', 'run_attempt': 1, 'langgraph_version': '1.2.12', 'langgraph_api_version': '0.17.0.dev3', 'langgraph_plan': 'enterprise', 'langgraph_host': 'self-hosted', 'langgraph_api_url': 'http://127.0.0.1:2024', 'run_id': '01a0f852-2437-7b23-ac74-f26e05568109', 'thread_id': '01a0f852-2437-7b23-ac74-f27e6db23df3', 'graph_id': 'agent', 'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'call_model', 'langgraph_triggers': ['branch:to:call_model'], 'langgraph_path': ['__pregel_pull', 'call_model'], 'langgraph_checkpoint_ns': 'call_model:3c9d4839-5b9d

Receiving new event of type: messages...
[{'content': 'voke', 'additional_kwargs': {}, 'response_metadata': {'model_provider': 'deepseek'}, 'type': 'AIMessageChunk', 'name': None, 'id': 'lc_run--01a0f852-281c-7401-af4f-eeea5aefc542', 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata': None, 'tool_call_chunks': [], 'chunk_position': None}, {'created_by': 'system', 'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca', 'run_attempt': 1, 'langgraph_version': '1.2.12', 'langgraph_api_version': '0.17.0.dev3', 'langgraph_plan': 'enterprise', 'langgraph_host': 'self-hosted', 'langgraph_api_url': 'http://127.0.0.1:2024', 'run_id': '01a0f852-2437-7b23-ac74-f26e05568109', 'thread_id': '01a0f852-2437-7b23-ac74-f27e6db23df3', 'graph_id': 'agent', 'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'call_model', 'langgraph_triggers': ['branch:to:call_model'], 'langgraph_path': ['__pregel_pull', 'call_model'], 'langgraph_checkpoint_ns': 'call_model:3c9d4839-5b

Receiving new event of type: messages...
[{'content': ' for', 'additional_kwargs': {}, 'response_metadata': {'model_provider': 'deepseek'}, 'type': 'AIMessageChunk', 'name': None, 'id': 'lc_run--01a0f852-281c-7401-af4f-eeea5aefc542', 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata': None, 'tool_call_chunks': [], 'chunk_position': None}, {'created_by': 'system', 'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca', 'run_attempt': 1, 'langgraph_version': '1.2.12', 'langgraph_api_version': '0.17.0.dev3', 'langgraph_plan': 'enterprise', 'langgraph_host': 'self-hosted', 'langgraph_api_url': 'http://127.0.0.1:2024', 'run_id': '01a0f852-2437-7b23-ac74-f26e05568109', 'thread_id': '01a0f852-2437-7b23-ac74-f27e6db23df3', 'graph_id': 'agent', 'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'call_model', 'langgraph_triggers': ['branch:to:call_model'], 'langgraph_path': ['__pregel_pull', 'call_model'], 'langgraph_checkpoint_ns': 'call_model:3c9d4839-5b

Receiving new event of type: messages...
[{'content': 'ctions', 'additional_kwargs': {}, 'response_metadata': {'model_provider': 'deepseek'}, 'type': 'AIMessageChunk', 'name': None, 'id': 'lc_run--01a0f852-281c-7401-af4f-eeea5aefc542', 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata': None, 'tool_call_chunks': [], 'chunk_position': None}, {'created_by': 'system', 'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca', 'run_attempt': 1, 'langgraph_version': '1.2.12', 'langgraph_api_version': '0.17.0.dev3', 'langgraph_plan': 'enterprise', 'langgraph_host': 'self-hosted', 'langgraph_api_url': 'http://127.0.0.1:2024', 'run_id': '01a0f852-2437-7b23-ac74-f26e05568109', 'thread_id': '01a0f852-2437-7b23-ac74-f27e6db23df3', 'graph_id': 'agent', 'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'call_model', 'langgraph_triggers': ['branch:to:call_model'], 'langgraph_path': ['__pregel_pull', 'call_model'], 'langgraph_checkpoint_ns': 'call_model:3c9d4839-

### Rest API

```bash
curl -s --request POST \
    --url "http://localhost:2024/runs/stream" \
    --header 'Content-Type: application/json' \
    --data "{
        \"assistant_id\": \"agent\",
        \"input\": {
            \"messages\": [
                {
                    \"role\": \"human\",
                    \"content\": \"What is LangGraph?\"
                }
            ]
        },
        \"stream_mode\": \"messages-tuple\"
    }"
```

## 后续步骤

现在你已经有了在本地运行的 LangGraph 应用, 接下来可以继续探索部署与高级功能, 把旅程走得更远:

* [部署快速入门](https://docs.langchain.com/langsmith/deployment-quickstart): 使用 LangSmith 部署你的 LangGraph 应用。
* [LangSmith](https://docs.langchain.com/langsmith/observability): 了解 LangSmith 的基础概念。
* [SDK Reference](https://reference.langchain.com/python/langsmith/deployment/sdk/): 探索 SDK API 参考文档。